In [1]:
!nvidia-smi

Tue May 26 19:24:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install langchain

In [3]:
!pip install langchain_community

In [4]:
!pip install langchain_text_splitter
!pip install langchain_chroma
!pip install langchain_groq

ERROR: Could not find a version that satisfies the requirement langchain_text_splitter (from versions: none)
ERROR: No matching distribution found for langchain_text_splitter


In [5]:
import os
from langchain_community.document_loaders import UnstructuredFileIOLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA

/tmp/ipykernel_5539/1563069720.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredFileIOLoader


In [6]:
# Import userdata to access secrets
from google.colab import userdata

# Retrieve the GROQ API key from Colab secrets
os.environ["GROQ_API_Key"] = userdata.get('GROQ_API_Key')
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')

In [7]:
import requests
url="https://learn.microsoft.com/en-us/azure/azure-functions/functions-reference-python?pivots=python-mode-decorators.pdf"
response=requests.get(url)
with open("Azure Functions developer reference guide for Python apps.pdf","wb") as f:
  f.write(response.content)

In [9]:
!pip install langsmith

In [8]:
import os
from langsmith import Client
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]="RAG_LLMOPS_PROJECT"
print('Lansmith tracing enabled')

Lansmith tracing enabled


In [10]:
!pip install 'unstructured[pdf]'

In [11]:
with open("Azure Functions developer reference guide for Python apps.pdf", "rb") as f:
    loader=UnstructuredFileIOLoader(f)

/tmp/ipykernel_5539/423903358.py:2: LangChainDeprecationWarning: The class `UnstructuredFileIOLoader` was deprecated in LangChain 0.2.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-unstructured package and should be used instead. To use it run `pip install -U `langchain-unstructured` and import as `from `langchain_unstructured import UnstructuredLoader``.
  loader=UnstructuredFileIOLoader(f)


In [12]:
text_splitter=CharacterTextSplitter(chunk_size=2000,chunk_overlap=400)

In [40]:
with open("Azure Functions developer reference guide for Python apps.pdf", "rb") as f:
    # Re-instantiate the loader with the open file object to ensure it's accessible
    file_loader = UnstructuredFileIOLoader(f)
    document = file_loader.load()

# Split the documents first
texts = text_splitter.split_documents(document)

# Now add the source to the metadata of each split document (chunk)
pdf_filename = "Azure Functions developer reference guide for Python apps.pdf"
for doc in texts:
    doc.metadata['source'] = pdf_filename

In [41]:
type(texts)

list

In [42]:
texts[2]

Document(metadata={'source': 'Azure Functions developer reference guide for Python apps.pdf'}, page_content='Use the azure-functions SDK and include type annotations to improve IntelliSense and editor support:\n\n# __init__.py\nimport azure.functions as func\n\ndef http_trigger(req: func.HttpRequest) -> str:\n\n# requirements.txt\nazure-functions\n\nThe azure-functions library\n\nThe azure-functions Python library provides the core types used to interact with the Azure Functions runtime. To see all types and methods available, visit the azure-functions API. Your function code can use azure-functions to:\n\nAccess trigger input data (for example, HttpRequest, TimerRequest)\n\nCreate output values (such as HttpResponse)\n\nInteract with runtime-provided context and binding data\n\nIf you\'re using azure-functions in your app, it must be included in your project dependencies.\n\nNote\n\nThe azure-functions library defines the programming surface for Python Azure Functions, but it isn’t a 

In [43]:
embeddings=HuggingFaceBgeEmbeddings()

/tmp/ipykernel_5539/3969363942.py:1: LangChainDeprecationWarning: Default values for HuggingFaceBgeEmbeddings.model_name were deprecated in LangChain 0.2.5 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceBgeEmbeddings constructor instead.
  embeddings=HuggingFaceBgeEmbeddings()


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [44]:
persist_directory='vector_db'

In [45]:
from google.colab import userdata
os.environ["GROQ_API_Key"] = userdata.get('GROQ_API_Key')

In [46]:
from langchain_groq import ChatGroq
import os

# Ensure the API key is available in the environment before initializing ChatGroq
# Assuming GROQ_API_Key is set correctly in a previous cell like fed4e2cb or y23hTq6I_Rgd
llm=ChatGroq(model='llama-3.3-70b-versatile', temperature=2, api_key=os.environ.get('GROQ_API_Key'))

In [47]:
vectordb=Chroma.from_documents(documents=texts,embedding=embeddings,persist_directory=persist_directory)

In [48]:
retriever=vectordb.as_retriever()

In [49]:
qa_chain=RetrievalQA.from_chain_type(llm=llm,chain_type="stuff",retriever=retriever, return_source_documents=True)

In [51]:
query='what are the functions from pdf'
response=qa_chain.invoke({'query':query})
print(response['result'])

The provided text is not a PDF, but rather a collection of passages about Azure Functions and Python programming. However, I can try to extract some function names from the text:

1. `http_trigger_with_cosmosdb`: This is a function that handles HTTP requests and interacts with a Cosmos DB database.
2. `func.HttpRequest`: This is a class that represents an HTTP request.
3. `func.HttpResponse`: This is a class that represents an HTTP response.
4. `func.DocumentList`: This is a class that represents a list of documents from a Cosmos DB database.
5. `func.Out`: This is a class that represents an output binding.
6. `func.FunctionApp`: This is a class that represents an Azure Functions app.

Additionally, the text mentions some function decorators, including:

1. `@app.function_name`: This decorator is used to define the name of a function.
2. `@app.route`: This decorator is used to define the route for a function.
3. `@app.cosmos_db_input`: This decorator is used to bind a function to a Cos

In [52]:
query='what are the python functions from pdf'
response=qa_chain.invoke({'query':query})
print(response['result'])

Based on the provided context, here are the Python functions mentioned:

1. **http_trigger**: This is a Python function defined with a decorator-based approach in `function_app.py`. It takes an HTTP request as input and returns an HTTP response.

    ```python
    @app.route(route="http_trigger")
    def http_trigger(req: func.HttpRequest) -> func.HttpResponse:
        return "Hello, World!"
    ```

2. **main**: This is another Python function defined in `__init__.py`. It also takes an HTTP request as input and returns an HTTP response.

    ```python
    def main(req: func.HttpRequest) -> func.HttpResponse:
        return func.HttpResponse("Hello, World!")
    ```

Both of these functions can be tested directly using Python by passing `None` as an argument.

```python
print(http_trigger(None))
print(main(None))
```


In [55]:
query='what are the deployment functions from pdf'
response=qa_chain.invoke({'query':query})
print(response['result'])

 According to the text, the following are the deployment mechanisms for Azure Functions:

1. **Azure Functions Core Tools**: Ideal for CI runs, local automation, or cross-platform work. Command: `func azure functionapp publish <APP_NAME>`.
2. **AZ CLI**: Useful for scripting deployments outside of Core Tools. Command: `az functionapp deployment source config-zip`.
3. **Visual Studio Code (Azure Functions Extension)**: Best for beginners or interactive deployments. Action: Command Palette → “Azure Functions: Deploy to Azure…”.
4. **GitHub Actions**: Ideal for GitHub-based CI/CD. Action: `Azure/functions-action@v1`.
5. **Azure Pipelines**:Best for enterprise CI/CD using Azure DevOps. Task: `AzureFunctionApp@2`.
6. **Custom Container Deployment**: Required for OS-level packages, custom Python builds, pinned runtimes, or unsupported dependencies. Command: `az functionapp create --image <container>`.
7. **Portal-based Function Creation**: Only for simple, dependency-free functions. Create f

In [58]:
import gradio as gr

def chatbot_response(question):

  try:
    response=qa_chain.invoke({'query':question})
    answer=response['result']
    sources=[]
    for doc in response['source_documents']:
      if 'source' in doc.metadata:
        sources.append(doc.metadata['source']) # Corrected key to 'source'
    source_text='\n'.join(list(set(sources)))
    final_response=f"""
{answer}
\n\nSources:\n{source_text}""" # Closed the f-string and added source_text
    return final_response # Added return statement
  except Exception as e:
    return f"An error occurred: {e}"

demo=gr.Interface(fn=chatbot_response,
                  inputs=gr.Textbox(
                      lines=2,
                      placeholder='Ask questions from your pdf'),
                  outputs="text",
                  title="PDF Chatbot")

demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8d9e529c8a69dfbd9a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://8d9e529c8a69dfbd9a.gradio.live
